# Distillation: Off-Policy Reasoning (SFT)

This notebook demonstrates off-policy distillation using supervised fine-tuning on the OpenThoughts3 dataset.

**Goal**: Transfer reasoning capabilities from a teacher model's traces to a student model.

**Method**: Fine-tune on pre-collected reasoning traces (off-policy = not generated during training).

In [ ]:
import os
os.environ['TINKER_API_KEY'] = 'YOUR_API_KEY_HERE'  # Replace with your key

## Dataset: OpenThoughts3

OpenThoughts3 contains 1.2M reasoning traces with:
- Step-by-step mathematical reasoning
- Chain-of-thought explanations
- Diverse problem types

This is **off-policy** because traces were generated by a teacher model beforehand.

In [ ]:
# Training configuration
config = {
    'model_name': 'meta-llama/Llama-3.2-1B',
    'dataset': 'openthoughts3',  # Pre-collected reasoning traces
    'learning_rate': 1e-3,       # Higher LR for LoRA
    'batch_size': 32,
    'lora_rank': 32,
}

print("Configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")

## Run Training

In [ ]:
!python -m tinker_cookbook.recipes.distillation.off_policy_reasoning \
    model_name="meta-llama/Llama-3.2-1B" \
    learning_rate=1e-3 \
    batch_size=32 \
    lora_rank=32

## Understanding Off-Policy vs On-Policy

| Aspect | Off-Policy (this notebook) | On-Policy |
|--------|---------------------------|----------|
| Data source | Pre-collected teacher traces | Generated during training |
| Method | Supervised fine-tuning | KL minimization |
| Compute | Lower (no generation) | Higher (needs sampling) |
| Adaptability | Fixed to dataset | Adapts to student |

## Key Metrics

| Metric | Description |
|--------|-------------|
| `train_mean_nll` | Training loss (lower = better) |
| `num_loss_tokens` | Tokens used for loss computation |
| `progress` | Training progress |

### Expected Results (AIME'24 benchmark)
- After 3000 steps with rank-128 LoRA: ~55% accuracy
- With full fine-tuning: similar results, lower LR needed

## Production Configuration

For best results on reasoning benchmarks:

In [ ]:
# Production config for Qwen3-8B
production_config = {
    'model_name': 'Qwen/Qwen3-8B-Base',
    'learning_rate': 1e-3,
    'batch_size': 128,
    'lora_rank': 128,
    'wandb_project': 'cookbook_distillation',
}

print("Production config:")
for k, v in production_config.items():
    print(f"  {k}: {v}")

## Analyze Results

In [ ]:
import json
import glob

# Find the latest metrics file
metrics_files = glob.glob('/home/ubuntu/tinker-examples/distillation/sft-*/metrics.jsonl')
if metrics_files:
    latest = max(metrics_files)
    print(f"Reading: {latest}\n")
    
    steps = []
    nlls = []
    
    with open(latest) as f:
        for i, line in enumerate(f):
            data = json.loads(line)
            if 'train_mean_nll' in data:
                steps.append(i)
                nlls.append(data['train_mean_nll'])
    
    if nlls:
        print(f"Steps trained: {len(steps)}")
        print(f"Starting NLL: {nlls[0]:.4f}")
        print(f"Current NLL: {nlls[-1]:.4f}")